In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import torch
torch.set_num_threads(4)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/bpic20_Rfp.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 147529,2017-02-14 15:34:34,UNKNOWN,organizational unit 65458,137.526306,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 147529,2017-02-14 15:34:43,UNKNOWN,organizational unit 65458,137.526306,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,9.0
2,request for payment 147529,2017-02-15 14:48:02,UNKNOWN,organizational unit 65458,137.526306,Request Payment,SYSTEM,UNDEFINED,83599.0
3,request for payment 147529,2017-02-20 17:32:08,UNKNOWN,organizational unit 65458,137.526306,Payment Handled,SYSTEM,UNDEFINED,441846.0
4,request for payment 147534,2017-03-02 15:55:43,UNKNOWN,organizational unit 65463,59.567024,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
5,request for payment 147534,2017-03-02 15:58:27,UNKNOWN,organizational unit 65463,59.567024,Request For Payment APPROVED by PRE_APPROVER,STAFF MEMBER,PRE_APPROVER,164.0
6,request for payment 147534,2017-03-02 16:07:38,UNKNOWN,organizational unit 65463,59.567024,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,551.0
7,request for payment 147534,2017-03-06 13:57:31,UNKNOWN,organizational unit 65463,59.567024,Request Payment,SYSTEM,UNDEFINED,337793.0
8,request for payment 147534,2017-03-13 17:31:05,UNKNOWN,organizational unit 65463,59.567024,Payment Handled,SYSTEM,UNDEFINED,617614.0
9,request for payment 147539,2017-03-06 14:40:07,UNKNOWN,organizational unit 65458,47.927757,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [3.00, 325445.40]                        57230.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [10.54, 665.70]                          74.7009    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
or

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR'}]

In [14]:
engine.branching_sets

[{'Request For Payment APPROVED by ADMINISTRATION',
  'Request For Payment APPROVED by BUDGET OWNER',
  'Request For Payment APPROVED by PRE_APPROVER',
  'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR',
  'Request For Payment SUBMITTED by EMPLOYEE'},
 {'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR'},
 {'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by MISSING',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED by SUPERVISOR'},
 {'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR'}]

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Rfp-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/200 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 176718,4,1,0,0.398020,0.312707,0.483333,0.577778,0.181818,...,0.417171,0.181818,0.083837,0.166667,1.008264e-03,0.333333,0.0,0.000000,0.0,0.0
1,0,request for payment 166711,5,1,0,0.305859,0.245052,0.366667,0.372222,0.307692,...,0.000000,0.307692,0.000000,0.000000,0.000000e+00,0.000000,0.0,0.000000,0.0,0.0
2,0,request for payment 184068,5,1,0,0.316012,0.182024,0.450000,0.488889,0.307692,...,0.306502,0.307692,0.084279,0.166667,1.892122e-03,0.222222,0.0,0.000000,0.0,0.0
3,0,request for payment 182365,5,1,0,0.376628,0.486588,0.266667,0.411111,0.307692,...,0.000000,0.307692,0.000000,0.000000,0.000000e+00,0.000000,0.0,0.000000,0.0,0.0
4,0,request for payment 168709,5,1,0,0.414522,0.370710,0.458333,0.461111,0.307692,...,0.194444,0.307692,0.083333,0.166667,0.000000e+00,0.111111,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,16,request for payment 178121,9,1,0,0.429556,0.391254,0.467857,0.566667,0.285714,...,0.497850,0.285714,0.164517,0.214286,1.147486e-01,0.333333,0.0,0.920198,0.0,1.0
131,16,request for payment 170670,9,1,0,0.427374,0.358320,0.496429,0.638095,0.285714,...,0.479855,0.285714,0.146521,0.142857,1.501857e-01,0.333333,0.0,0.860937,0.0,1.0
132,16,request for payment 171283,9,1,0,0.381503,0.280863,0.482143,0.528571,0.285714,...,0.309129,0.285714,0.118653,0.071429,1.658769e-01,0.190476,0.0,0.880432,0.0,1.0
133,16,request for payment 171342,9,1,0,0.383064,0.248272,0.517857,0.550000,0.285714,...,0.214286,0.285714,0.071429,0.142857,7.531038e-07,0.142857,0.0,0.833629,0.0,1.0


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()